# [Introduction](https://learn.microsoft.com/en-us/training/paths/get-started-querying-with-transact-sql/)

## Overview

When retrieving data from a database using a **Transact-SQL (T-SQL)** `SELECT` statement, it is rarely enough to just dump all the raw data. You typically need to organize the data into a logical sequence or isolate specific rows that meet defined business criteria.

The two primary clauses used for refining your query results are:

* **Sorting:** Handled by the `ORDER BY` clause.
* **Filtering:** Handled by the `WHERE` clause.

---

## Query Process Flow

Below is a Mermaid flowchart illustrating the high-level logical progression of retrieving, filtering, and sorting data in a database query.


```mermaid
graph TD
    A[(Database)] -->|SELECT Statement| B(Retrieve Raw Data)
    B --> C{Criteria to Filter?}
    C -->|Yes| D[Apply WHERE Clause\nFilters rows based on predicates]
    C -->|No| E{Order to Sort?}
    D --> E
    E -->|Yes| F[Apply ORDER BY Clause\nSorts ascending or descending]
    E -->|No| G((Final Query Results))
    F --> G

```

---

## Learning Objectives

In this module, you will learn how to manipulate your result sets using the following techniques:

| Operation | Description | Associated Concept |
| --- | --- | --- |
| **Sort Results** | Arrange the output of a query systematically (e.g., alphabetically, numerically). | `ORDER BY` |
| **Limit Results** | Restrict the sorted output to display only the top *n* rows. | `TOP` |
| **Page Results** | Segment large datasets into smaller, manageable "pages" for application display. | `OFFSET` / `FETCH` |
| **Remove Duplicates** | Clean your output by ensuring only unique rows are returned. | `DISTINCT` |
| **Filter Data** | Apply specific logical predicates to include only the exact rows you need. | `WHERE` |



# 📘 [Sorting Query Results with `ORDER BY`](https://learn.microsoft.com/en-us/training/modules/sort-filter-queries/2-sort-your-data-as-its-returned)


## Overview: The Importance of Sorting
In relational database theory, a table is an unordered set of rows. SQL Server **does not guarantee** the physical order of rows in a table. Therefore, if you want to control the exact order in which rows are returned to the client application, you **must** use the `ORDER BY` clause. 

In the logical order of query processing, `ORDER BY` is the **very last phase** to be executed.

### Logical Query Processing Flow
Because `ORDER BY` is processed last, it has access to all the columns and aliases defined earlier in the query.

```mermaid
graph TD
    A[1. FROM: Identify Tables] --> B[2. WHERE: Filter Rows]
    B --> C[3. SELECT: Choose Columns & Define Aliases]
    C --> D[4. ORDER BY: Sort Final Output]
    
    D --> E((Rows returned to Client))
    
    style D fill:#e8f5e9,stroke:#2e7d32,stroke-width:3px
    style C fill:#fff3e0,stroke:#ef6c00,stroke-width:2px
    style E fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px
```

---

## 1. Basic Syntax and Sort Direction

To sort results, append the `ORDER BY` clause at the end of your `SELECT` statement. 

*   **`ASC`**: Ascending order (A-Z, 0-9). This is the **default** if omitted.
*   **`DESC`**: Descending order (Z-A, 9-0).
*   You can specify a different direction for *each* column in a multi-column sort.

```sql
-- Basic Syntax
SELECT <select_list>
FROM <table_source>
ORDER BY <order_by_list> [ASC | DESC];

-- Example: Sort by Category (Ascending), then by ListPrice (Descending)
SELECT ProductCategoryID AS Category, Name AS ProductName, ListPrice
FROM Production.Product
ORDER BY Category ASC, ListPrice DESC;
```

---

## 2. What can you sort by? (4 Rules)

The `ORDER BY` clause is highly flexible. You can sort by four different types of elements:

### Rule 1: Columns by Name
Sort by the actual column names from the source tables. Results are sorted by the first column, then sub-sorted by the next.

### Rule 2: Column Aliases
Because `ORDER BY` is processed **after** `SELECT`, you can use the aliases you created in the `SELECT` list.

### Rule 3: Ordinal Position (Column Number)
You can sort by the position of the column in the `SELECT` list (e.g., `1`, `2`, `3`). 
*⚠️ **Best Practice:** Avoid this in production code due to poor readability and maintenance risks. It is mostly useful for quick troubleshooting.*

### Rule 4: Columns NOT in the SELECT List
You can sort by a column that exists in the `FROM` table but wasn't included in the `SELECT` output. 
*🚨 **CRITICAL EXCEPTION:** If your query uses the `DISTINCT` keyword, any column in the `ORDER BY` list **MUST** also be included in the `SELECT` list.*

---

```sql
-- RULE 1 & 2: Sorting by Column Name and Alias
SELECT 
    ProductID, 
    Name AS ProductName, 
    ListPrice AS Price
FROM Production.Product
ORDER BY ProductName ASC, ProductID DESC; 
-- (Using alias 'ProductName' and column 'ProductID')

-- RULE 3: Sorting by Ordinal Position
-- Sorts by 1st column (ProductID), then 3rd column (ListPrice)
SELECT 
    ProductID, 
    Name, 
    ListPrice
FROM Production.Product
ORDER BY 1 ASC, 3 DESC; 

-- RULE 4: Sorting by a column NOT in the SELECT list
SELECT 
    Name, 
    ListPrice
FROM Production.Product
ORDER BY ProductID ASC; 
-- (ProductID is not selected, but used for sorting)

-- 🚨 THE DISTINCT EXCEPTION: This will FAIL!
-- SELECT DISTINCT Name, ListPrice
-- FROM Production.Product
-- ORDER BY ProductID; 
-- ERROR: ORDER BY items must appear in the select list if SELECT DISTINCT is specified.
```

---

## 3. The `DISTINCT` Constraint Flowchart

To help visualize why the `DISTINCT` exception exists, review the logic below. `DISTINCT` removes duplicate rows *before* sorting. If you sort by a column that isn't selected, the database wouldn't know how to handle duplicate values in the selected columns that have different values in the unselected sorting column.

```mermaid
flowchart TD
    Start[Query uses ORDER BY] --> CheckDistinct{Is DISTINCT used?}
    
    CheckDistinct -->|No| AllowAll[Allowed: Can sort by ANY column in FROM tables]
    CheckDistinct -->|Yes| CheckSelect{Is the ORDER BY column in the SELECT list?}
    
    CheckSelect -->|Yes| AllowDistinct[Allowed: Sort by selected columns only]
    CheckSelect -->|No| Error[❌ ERROR: ORDER BY items must appear in the select list]
    
    style Start fill:#f9f2f4,stroke:#333
    style CheckDistinct fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style CheckSelect fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style Error fill:#ffebee,stroke:#c62828,stroke-width:2px
    style AllowAll fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
    style AllowDistinct fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
```

---

## Summary Checklist
- [x] `ORDER BY` is the **last** step in logical query processing.
- [x] Without `ORDER BY`, row return order is **not guaranteed**.
- [x] `ASC` is default; `DESC` reverses the order.
- [x] You can sort by column names, aliases, ordinal numbers, or unselected columns.
- [x] **Exception:** If using `DISTINCT`, you can *only* sort by columns explicitly listed in the `SELECT` clause.

***

# [Limiting Sorted Results with the `TOP` Clause](https://learn.microsoft.com/en-us/training/modules/sort-filter-queries/3-use-top-clause-to-limit-rows)

## Overview
The `TOP` clause is a **Microsoft-proprietary** extension of the `SELECT` clause in T-SQL (SQL Server). It allows you to limit the number or percentage of rows returned by a query. 

### Key Characteristics:
- **Proprietary**: Specific to T-SQL (not standard ANSI SQL).
- **Flexible Limits**: Accepts a positive integer, a percentage, a constant, or an expression.
- **Sorting Dependency**: Most effective when paired with an `ORDER BY` clause. Without it, the rows returned are unpredictable.
- **Best Practice**: Always use TOP in conjunction with an ORDER BY clause to ensure meaningful and predictable results.

***

## 1. Basic Syntax and Logical Processing

The simplified syntax for using `TOP` with `ORDER BY` is:

```sql
SELECT TOP (N) <column_list>
FROM <table_source>
WHERE <search_condition>
ORDER BY <order_list> [ASC|DESC];
```

### Logical Execution Flow
It is crucial to understand that `TOP` filters the data **after** it has been sorted. 

```mermaid
graph TD
    A[(Source Data)] --> B{Is ORDER BY included?}
    B -- No --> C[⚠️ Unpredictable Output Order]
    B -- Yes --> D[Data Sorted Meaningfully]
    C --> E{Apply TOP Filter}
    D --> E
    E -- Fixed Integer --> F[Select TOP N Rows]
    E -- Percentage --> G[Select TOP N% Rows <br> *Rounds up to nearest integer*]
    F --> H{WITH TIES specified?}
    G --> H
    H -- No --> I[Strict Cutoff]
    H -- Yes --> J[Include additional rows matching <br> the value of the final row]
    I --> K([Final Result Set])
    J --> K
```


***

**Basic TOP Example**
```sql
-- Retrieve the 10 most expensive products
-- ORDER BY is critical here to ensure we get the "most expensive" and not just "any 10"

SELECT TOP 10 Name, ListPrice
FROM Production.Product
ORDER BY ListPrice DESC;
```

***

## 2. Using `WITH TIES`

By default, `TOP N` strictly returns exactly `N` rows. However, if the Nth row has the same sorting value as subsequent rows, those tied rows are excluded. 

The **`WITH TIES`** option tells SQL Server to include any additional rows that tie with the last row of the top N results.

### How `WITH TIES` Works:
```mermaid
flowchart TD
    classDef step fill:#e8f5e9,stroke:#388e3c,stroke-width:2px;
    classDef decision fill:#fff3e0,stroke:#f57c00,stroke-width:2px;
    
    A["Sort Data via ORDER BY"] --> B["Identify the Top N rows"]:::step
    B --> C["Find the value of the Nth row"]:::step
    C --> D{"Do subsequent rows have the SAME value?"}:::decision
    D -->|Yes| E["Include them (WITH TIES)"]:::step
    D -->|No| F["Stop and return results"]:::step
```

*Use `WITH TIES` when your business logic requires all tied records to be included, ensuring no data is arbitrarily cut off.*


***

**WITH TIES Example**
```sql
-- Retrieve the top 10 most expensive products, 
-- INCLUDING any other products that share the same price as the 10th product.

SELECT TOP 10 WITH TIES Name, ListPrice
FROM Production.Product
ORDER BY ListPrice DESC;
```

***

## 3. Using `PERCENT`

Instead of a fixed number of rows, you can return a **percentage** of the eligible rows by using the `PERCENT` keyword.

### Important Rules for `PERCENT`:
1. It calculates the percentage based on the total number of rows that qualify after the `WHERE` clause.
2. **Rounding**: SQL Server will **round up** to the nearest integer for the final row count.
3. It can be combined with `WITH TIES`.

```sql
-- Syntax for PERCENT
SELECT TOP (N) PERCENT <column_list>
FROM <table_source>
ORDER BY <order_list>;
```


***

**PERCENT Example**
```sql
-- Retrieve the top 10% most expensive products
-- If 10% equals 29.5 rows, SQL Server will round up and return 30 rows.

SELECT TOP 10 PERCENT Name, ListPrice
FROM SalesLT.Product
ORDER BY ListPrice DESC;
```

***

## 4. Important Considerations & Limitations

While `TOP` is widely used by SQL Server professionals for limiting results, keep the following facts in mind:

1. **Proprietary Syntax**: `TOP` is specific to T-SQL. If you need cross-platform compatibility (e.g., PostgreSQL, MySQL, Oracle), use the ANSI standard `LIMIT` or `FETCH FIRST` clauses instead.
2. **No Row Skipping**: `TOP` on its own **cannot skip rows** (it doesn't support an `OFFSET`). To get rows 11-20, you would need to use `OFFSET ... FETCH` or window functions like `ROW_NUMBER()`.
3. **Coupled Sorting due to ORDER BY Dependency**: Because `TOP` relies on `ORDER BY` to establish precedence, **you cannot use one sort order to filter the rows via `TOP` and a completely different sort order for the final output**. The `ORDER BY` clause dictates both the filtering and the final presentation order.
4. **Best Practice**: Always use TOP in conjunction with an ORDER BY clause to ensure meaningful and predictable results.

### Summary Table
| Feature | Description | Example |
| :--- | :--- | :--- |
| **Standard TOP** | Returns exact number of rows | `TOP 10` |
| **WITH TIES** | Includes rows tied with the Nth row | `TOP 10 WITH TIES` |
| **PERCENT** | Returns a % of total rows (rounds up) | `TOP 10 PERCENT` |
```

# [Paging Results with `OFFSET-FETCH`](https://learn.microsoft.com/en-us/training/modules/sort-filter-queries/4-page-results)

## Overview
The `OFFSET-FETCH` clause is an extension of the `ORDER BY` clause that allows you to return a specific **range** (or "page") of rows from your query results. It is the standard, ANSI-compliant method for implementing pagination in SQL Server.

### Key Concepts:
- **Offset**: The starting point (how many rows to *skip*).
- **Fetch**: The number of rows to *return* after the offset.
- **Client-Side State**: Each query with `OFFSET-FETCH` runs **independently**. The server does not maintain state between queries. You must track your current page/position using client-side code (e.g., in your application).

## 1. Syntax and Execution Flow

The `OFFSET-FETCH` clause is technically part of the `ORDER BY` clause. 

### Standard Syntax:
```sql
ORDER BY <column_list>
OFFSET { integer_constant | expression } { ROW | ROWS }
[FETCH { FIRST | NEXT } { integer_constant | expression } { ROW | ROWS } ONLY];
```

### Logical Processing Flow:
```mermaid
graph TD
    classDef default fill:#f9f9f9,stroke:#333,stroke-width:2px;
    classDef skip fill:#ffebee,stroke:#c62828,stroke-width:2px;
    classDef fetch fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    
    A["1. FROM & WHERE"] --> B["2. ORDER BY (Sort Data)"]
    B --> C["3. OFFSET N ROWS"]:::skip
    C -->|"Skip the first N rows"| D["4. FETCH NEXT M ROWS ONLY"]:::fetch
    D -->|"Return the next M rows"| E["5. Final Result Set"]
    
    note1["Must be used with ORDER BY"]
    B -.-> note1
```

**Example: Fetching Page 1**
```sql
-- PAGE 1: Get the first 10 most expensive products
-- We skip 0 rows and fetch the next 10.

SELECT ProductID, ProductName, ListPrice
FROM Production.Product
ORDER BY ListPrice DESC 
OFFSET 0 ROWS            -- Skip zero rows (start at the beginning)
FETCH NEXT 10 ROWS ONLY; -- Get the next 10 rows
```

***


**Example: Fetching Page 2**
```sql
-- PAGE 2: Get the next 10 most expensive products
-- We skip the first 10 rows and fetch the next 10.

SELECT ProductID, ProductName, ListPrice
FROM Production.Product
ORDER BY ListPrice DESC 
OFFSET 10 ROWS           -- Skip the first 10 rows
FETCH NEXT 10 ROWS ONLY; -- Get the next 10 rows
```

***

## 2. Rules and Keyword Flexibility

The syntax is designed to be readable and flexible. Here are the rules to remember:

| Clause | Required? | Notes |
| :--- | :---: | :--- |
| **`OFFSET`** | ✅ **Yes** | You must specify an offset, even if it is `0`. |
| **`FETCH`** | ❌ **No** | If omitted, the query returns **all** remaining rows after the offset. |

### Interchangeable Keywords
To make the syntax read more like natural English, SQL Server allows you to swap certain keywords without changing the behavior:
- `ROW` and `ROWS` are completely interchangeable.
- `FIRST` and `NEXT` are completely interchangeable.

**Examples of valid, identical syntax:**
```sql
-- All of these do the exact same thing:
OFFSET 10 ROWS FETCH NEXT 10 ROWS ONLY
OFFSET 10 ROW FETCH NEXT 10 ROW ONLY
OFFSET 10 ROWS FETCH FIRST 10 ROWS ONLY
```
## 3. Crucial Best Practice: Deterministic Ordering

When paging through data, your `ORDER BY` clause **must** yield a unique, deterministic result. 

### The Problem:
If you sort by a column that contains duplicate values (e.g., `ListPrice`), the SQL Server query optimizer might place tied rows on different pages during different executions. A row could accidentally appear on Page 1 and Page 2, or be skipped entirely!

### The Solution:
Always include a **unique tie-breaker** (like a Primary Key) in your `ORDER BY` clause to guarantee deterministic pagination.

```mermaid
flowchart TD
    classDef bad fill:#ffcdd2,stroke:#b71c1c,stroke-width:2px;
    classDef good fill:#c8e6c9,stroke:#1b5e20,stroke-width:2px;
    
    A["Query: ORDER BY ListPrice DESC"] --> B{"Are there duplicate prices?"}
    B -->|Yes| C["❌ Non-Deterministic"]:::bad
    C --> D["Optimizer might split tied rows across Page 1 and Page 2"]
    
    B -->|No| E["✅ Deterministic"]:::good
    E --> F["Strict, predictable paging"]
    
    D --> G["🛠️ Fix: Add a unique column to ORDER BY"]
    G --> H["ORDER BY ListPrice DESC, ProductID ASC"]:::good
```

**Example: Implementing Deterministic Paging**

```sql
-- BEST PRACTICE: Add ProductID as a tie-breaker to ensure deterministic results.
-- This guarantees that products with the exact same ListPrice 
-- will always appear in the exact same order on every page.

SELECT ProductID, ProductName, ListPrice
FROM Production.Product
ORDER BY ListPrice DESC, ProductID ASC -- Unique ordering guaranteed!
OFFSET 10 ROWS 
FETCH NEXT 10 ROWS ONLY;
```

***

## Summary

- Use **`OFFSET-FETCH`** for paging results (it is the modern, ANSI-standard replacement for older T-SQL `TOP` workarounds).
- **`OFFSET`** is mandatory; **`FETCH`** is optional.
- Always use **deterministic sorting** (include a unique key in `ORDER BY`) to prevent rows from shifting between pages.
- Remember that the server is stateless; your application must calculate and pass the correct `OFFSET` value for each page request.


# [Removing Duplicates with `DISTINCT`](https://learn.microsoft.com/en-us/training/modules/sort-filter-queries/5-remove-duplicates)

## Why Do Duplicates Occur in Queries?
Even if the rows in a base table are strictly unique (e.g., enforced by a Primary Key), selecting only a **subset of columns** can result in duplicate rows in your final output. 

### Example Scenario:
Imagine a `Production.Supplier` table where each supplier has a unique `SupplierID`. However, multiple suppliers might be located in the exact same city. If you only query the location columns, the resulting rows will no longer be unique.

```mermaid
graph TD
    A[(Source Data <br> Unique Rows)] --> B[Select Subset of Columns]
    B --> C{Implicit or Explicit Keyword?}
    C -- Default / ALL --> D[Return every row processed]
    C -- DISTINCT --> E[Evaluate row combinations]
    D --> F([Result Set: Contains Duplicates])
    E --> G[Keep only the first instance of <br> each unique combination]
    G --> H([Result Set: Strictly Unique Rows])
    
    style D fill:#f9d0c4,stroke:#333,stroke-width:2px
    style G fill:#d4edda,stroke:#333,stroke-width:2px
```

```sql
-- This query will return duplicate rows if multiple suppliers share a city
SELECT City, CountryRegion
FROM Production.Supplier
ORDER BY CountryRegion, City;
```



***

## Implicit `ALL` vs. Explicit `DISTINCT`

By default, the `SELECT` clause includes an implicit **`ALL`** keyword, which tells SQL Server to return every row that matches the query, including duplicates. 

To remove duplicate rows from your result set, you must explicitly use the **`DISTINCT`** keyword.

### How `DISTINCT` Works:
`DISTINCT` evaluates the **entire combination** of columns in your `SELECT` list. It will return exactly one row for every unique combination of values.

```mermaid
graph TD
    classDef table fill:#f9f9f9,stroke:#333,stroke-width:2px;
    classDef all fill:#ffebee,stroke:#c62828,stroke-width:2px;
    classDef distinct fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;

    A["Base Table: Production.Supplier\n(Unique rows via SupplierID)"] --> B["SELECT City, CountryRegion"]
    
    B --> C["SELECT ALL\nReturns every row\n(e.g., Burnaby, Canada appears 3 times)"]:::all
    B --> D["SELECT DISTINCT\nFilters out duplicate combinations\n(e.g., Burnaby, Canada appears 1 time)"]:::distinct
    
    C --> E["Result: Duplicates Included"]
    D --> F["Result: Unique Combinations Only"]
```
***

**Example: Querying with Implicit ALL**
```sql
-- Using implicit ALL (or explicitly writing ALL)
-- This returns all rows, resulting in duplicate City/Country combinations 
-- (e.g., Brossard, Burnaby, and Calgary appear multiple times).

SELECT ALL City, CountryRegion
FROM Production.Supplier
ORDER BY CountryRegion, City;
```

***

**Example: Querying with DISTINCT**
```sql
-- Using DISTINCT
-- This removes duplicate rows, returning only one instance 
-- of each unique City and CountryRegion combination.

SELECT DISTINCT City, CountryRegion
FROM Production.Supplier
ORDER BY CountryRegion, City;
```

***

## Key Takeaways & Best Practices

1. **Scope of DISTINCT**: `DISTINCT` applies to the *entire row* of the result set, not individual columns. If you select `City` and `CountryRegion`, it looks for unique *pairs* of those two values.
2. **Handling NULLs**: `DISTINCT` treats all `NULL` values as equal. If multiple rows have `NULL` in the selected columns, only one row with `NULL` will be returned.
3. **Performance Impact**: Removing duplicates requires the database engine to sort or hash the result set in memory/tempdb. This adds overhead, so only use `DISTINCT` when you specifically need unique values.
4. **ORDER BY Rule**: When using `DISTINCT`, any column you use in the `ORDER BY` clause **must** also be included in the `SELECT` list. SQL Server cannot sort by a column that has been filtered out of the final distinct result set.




# [Filtering Data with Predicates](https://learn.microsoft.com/en-us/training/modules/sort-filter-queries/6-filter-data)

## Overview
While a basic `SELECT` statement evaluates every row in a table, the **`WHERE`** clause allows you to define conditions that filter the data. This reduces the result set to only the rows you need.

```mermaid
graph TD
    A[(Table Data)] --> B[Row 1]
    A --> C[Row 2]
    A --> D[Row N...]
    
    B --> E{Evaluate Predicate<br>e.g., Price < 10}
    C --> E
    D --> E
    
    E -- TRUE --> F([Include in Result Set])
    E -- FALSE --> G[Discard]
    E -- UNKNOWN --> G
    
    style F fill:#d4edda,stroke:#333,stroke-width:2px
    style G fill:#f9d0c4,stroke:#333,stroke-width:2px
```

### What is a Predicate?
A predicate is a condition within the `WHERE` clause that evaluates to **TRUE**, **FALSE**, or **UNKNOWN** for each row. Only rows that evaluate to **TRUE** are returned.

### Basic Comparison Operators
The most common way to build predicates is using basic comparison operators:
- `=` (Equals)
- `<>` or `!=` (Not equals)
- `>` (Greater than)
- `>=` (Greater than or equal to)
- `<` (Less than)
- `<=` (Less than or equal to)


***

**Example: Basic Operators & NULL Handling**
```sql
-- 1. Basic Equality (=)
-- Returns products in category 2
SELECT ProductCategoryID AS Category, ProductName
FROM Production.Product
WHERE ProductCategoryID = 2;

-- 2. Less Than (<)
-- Returns products costing less than $10.00
SELECT ProductCategoryID AS Category, ProductName
FROM Production.Product
WHERE ListPrice < 10.00;

-- 3. Handling NULLs (IS NULL / IS NOT NULL)
-- You cannot use = NULL. You must use IS NULL or IS NOT NULL.
SELECT ProductCategoryID AS Category, ProductName
FROM Production.Product
WHERE ProductName IS NOT NULL;
```

***

## Combining Conditions: `AND` / `OR`

You can combine multiple predicates using logical operators:
- **`AND`**: Both conditions must be TRUE.
- **`OR`**: At least one condition must be TRUE.

### ⚠️ Crucial Rule: Operator Precedence
In T-SQL, **`AND` operators are processed before `OR` operators**. If you mix them without parentheses, you might get unexpected results. 

**Best Practice:** *Always* use parentheses when combining more than two predicates to explicitly define the evaluation order.

### Visualizing Precedence
```mermaid
flowchart TD
    classDef default fill:#f9f9f9,stroke:#333,stroke-width:2px;
    classDef and fill:#bbdefb,stroke:#1565c0,stroke-width:2px;
    classDef or fill:#ffe0b2,stroke:#ef6c00,stroke-width:2px;
    classDef paren fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px;

    subgraph Implicit [❌ Implicit Precedence: AND before OR]
        direction LR
        I1["Category = 2"] --> I_AND["AND"]:::and
        I2["Category = 3"] --> I_AND
        I_AND --> I_OR["OR"]:::or
        I3["Price < 10.00"] --> I_OR
        I_OR --> I_Res["Evaluates as: (Cat=2 AND Cat=3) OR Price<10"]
    end

    subgraph Explicit [✅ Explicit Precedence: Using Parentheses]
        direction LR
        E1["Category = 2"] --> E_OR["OR"]:::or
        E2["Category = 3"] --> E_OR
        E_OR --> E_Paren["( )"]:::paren
        E_Paren --> E_AND["AND"]:::and
        E3["Price < 10.00"] --> E_AND
        E_AND --> E_Res["Evaluates as: (Cat=2 OR Cat=3) AND Price<10"]
    end
```


***

**Example: Multiple Conditions**
```sql
-- WITHOUT PARENTHESES (Implicit AND before OR)
-- This actually looks for products that are in BOTH category 2 AND 3 (impossible) 
-- OR cost less than 10.00.
SELECT ProductCategoryID AS Category, ProductName
FROM Production.Product
WHERE ProductCategoryID = 2 OR ProductCategoryID = 3
    AND ListPrice < 10.00;

-- WITH PARENTHESES (Best Practice)
-- This correctly filters for products in category 2 OR 3, 
-- AND ensures they cost less than 10.00.
SELECT ProductCategoryID AS Category, ProductName
FROM Production.Product
WHERE (ProductCategoryID = 2 OR ProductCategoryID = 3)
    AND (ListPrice < 10.00);
```

***

## Simplifying Predicates: `IN` and `BETWEEN`

### The `IN` Operator
`IN` is a clean shortcut for multiple `OR` equality conditions on the same column. It does not negatively impact performance.
```sql
-- Instead of: WHERE Cat = 2 OR Cat = 3 OR Cat = 4
-- Use:
WHERE ProductCategoryID IN (2, 3, 4);
```

### The `BETWEEN` Operator
`BETWEEN` filters for an upper and lower bound. 
**Important:** `BETWEEN` is **inclusive**. It includes the boundary values.

#### ⚠️ The Date/Time Trap
When querying `DATETIME` columns, if you don't specify the time, SQL Server defaults to `00:00:00.000`. A `BETWEEN` query ending on `'2012-12-31'` will **exclude** any records modified later in the day on Dec 31st!

**Solutions for Date Ranges:**
1. Include the exact end time: `BETWEEN '2012-01-01 00:00:00' AND '2012-12-31 23:59:59.999'`
2. Use `>=` and `<` (Recommended Best Practice): `>= '2012-01-01' AND < '2013-01-01'`


***

**IN and BETWEEN Examples**
```sql
-- Using IN
SELECT ProductCategoryID AS Category, ProductName
FROM Production.Product
WHERE ProductCategoryID IN (2, 3, 4);

-- Using BETWEEN (Inclusive)
SELECT ProductCategoryID AS Category, ProductName
FROM Production.Product
WHERE ListPrice BETWEEN 1.00 AND 10.00;

-- ❌ BAD: Misses records modified after midnight on Dec 31st
SELECT ProductName, ModifiedDate
FROM Production.Product
WHERE ModifiedDate BETWEEN '2012-01-01' AND '2012-12-31';

-- ✅ GOOD: Safely captures the entire year of 2012
SELECT ProductName, ListPrice, ModifiedDate
FROM Production.Product
WHERE ModifiedDate >= '2012-01-01' 
  AND ModifiedDate < '2013-01-01';
```

***

## Pattern Matching with `LIKE`

The `LIKE` operator is exclusively used for character data and supports **wildcard characters for partial string matching**.

### Wildcard Characters:
| Wildcard | Description | Example | Matches |
| :---: | :--- | :--- | :--- |
| **`%`** | Any string of **0 or more** characters | `'%mountain%'` | "Mountain Bike", "HL Mountain Frame" |
| **`_`** | Exactly **one** single character | `'Socks, _'` | "Socks, M", "Socks, L" |
| **`[ ]`** | A **single character** within a specified range or set | `'[0-9]'` | Any single digit (0 through 9) |

### Complex Patterns
You can combine these to build highly specific string patterns. For example, matching a specific product naming convention:

```sql
SELECT ProductName, ListPrice
FROM SalesLT.Product
WHERE ProductName LIKE 'Mountain-[0-9][0-9][0-9] %, [0-9][0-9]';
```

*(Matches "Mountain-", followed by 3 digits, a space, any string, a comma, a space, and 2 digits).*


***

**LIKE Operator Examples**
```sql
-- 1. % Wildcard: Contains the word "mountain" anywhere
SELECT Name, ListPrice
FROM SalesLT.Product
WHERE Name LIKE '%mountain%';

-- 2. _ Wildcard: Ends with a single character after "Socks, "
SELECT ProductName, ListPrice
FROM SalesLT.Product
WHERE ProductName LIKE 'Mountain Bike Socks, _';

-- 3. [ ] Wildcard: Complex pattern matching
-- Starts with "Mountain-", 3 digits, space, any text, comma, space, 2 digits
SELECT ProductName, ListPrice
FROM SalesLT.Product
WHERE ProductName LIKE 'Mountain-[0-9][0-9][0-9] %, [0-9][0-9]';
```

***

## Summary of Filtering Data

- **`WHERE`** clause filters rows based on predicates that evaluate to TRUE.
- Use **`IS NULL`** or **`IS NOT NULL`** to check for missing data (never use `= NULL`).
- Combine conditions with **`AND`** / **`OR`**. Always use **parentheses** to control evaluation order.
- **`IN`** simplifies multiple `OR` equality checks.
- **`BETWEEN`** is inclusive. Be careful with `DATETIME` boundaries; using `>=` and `<` is often safer.
- **`LIKE`** enables pattern matching using `%` (multiple chars), `_` (single char), and `[ ]` (char ranges).



# [Exercise: Sorting, Filtering, and Paging Query Results](https://microsoftlearning.github.io/dp-080-Transact-SQL/Instructions/Labs/02-filter-sort.html)

In this comprehensive exercise, we will apply the concepts of sorting, limiting, paging, and filtering data using the `AdventureWorks` database. 

### What we will cover:
1. **Sorting** results with `ORDER BY`
2. **Limiting** results with `TOP`
3. **Paging** results with `OFFSET-FETCH`
4. **Removing duplicates** with `DISTINCT`
5. **Filtering** data with the `WHERE` clause and various operators

***

## 1. Sorting Results with `ORDER BY`

By default, SQL Server returns rows in an arbitrary order. To control the output order, we use the `ORDER BY` clause.

### Key Rules:
- **Default Sort**: Ascending (`ASC`).
- **Descending Sort**: Explicitly use `DESC`.
- **Multiple Columns**: You can sort by multiple columns. The database sorts by the first column, and then uses the second column to break ties.

***

**Example: Basic Sorting**
```sql
-- 1. Unsorted (Arbitrary order)
SELECT Name, ListPrice
FROM SalesLT.Product;

-- 2. Sorted by Name (Ascending is default)
SELECT Name, ListPrice
FROM SalesLT.Product
ORDER BY Name;

-- 3. Sorted by ListPrice (Ascending)
SELECT Name, ListPrice
FROM SalesLT.Product
ORDER BY ListPrice;

-- 4. Sorted by ListPrice (Descending - Most expensive first)
SELECT Name, ListPrice
FROM SalesLT.Product
ORDER BY ListPrice DESC;
```

***

### Sorting by Multiple Columns
When sorting by multiple columns, SQL Server applies a **hierarchical sort**. It first sorts by the primary column. If there are ties (duplicate values) in the primary column, it uses the secondary column to determine the order of those specific tied rows.

```mermaid
flowchart TD
    classDef primary fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef secondary fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    
    A["1. Sort by ListPrice DESC"]:::primary
    A --> B{"Are there tied prices?"}
    B -->|No| C["Return sorted by Price"]
    B -->|Yes| D["2. Sort tied rows by Name ASC"]:::secondary
    D --> E["Return sorted by Price, then Name"]
```


***

**Example: Multiple Column Sorting**
```sql
-- Sort by ListPrice DESC, then by Name ASC for products with the same price
SELECT Name, ListPrice
FROM SalesLT.Product
ORDER BY ListPrice DESC, Name ASC;
```

***

## 2. Restricting Results with `TOP`

The `TOP` clause limits the number of rows returned. It is highly recommended to always pair `TOP` with an `ORDER BY` clause; otherwise, you are just getting an arbitrary slice of data.

### Variations of TOP:
- **`TOP (N)`**: Returns exactly N rows.
- **`TOP (N) WITH TIES`**: Returns N rows, *plus* any additional rows that tie with the Nth row's sorting value.
- **`TOP (N) PERCENT`**: Returns a percentage of the total qualifying rows.

***

**Example: Using TOP**
```sql
-- Basic TOP: Get the 20 most expensive products
SELECT TOP (20) Name, ListPrice
FROM SalesLT.Product
ORDER BY ListPrice DESC;
```

***

### How `WITH TIES` Works
If the 20th most expensive product costs $100, and there are two other products that also cost exactly $100, `WITH TIES` will include them, resulting in 22 rows instead of 20.

```mermaid
flowchart TD
    classDef top fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef tie fill:#fff3e0,stroke:#f57c00,stroke-width:2px;
    classDef cut fill:#ffebee,stroke:#c62828,stroke-width:2px;

    subgraph TOP20["✅ Standard TOP 20"]
        direction TB
        A["Row 18: $150"]:::top
        B["Row 19: $120"]:::top
        C["Row 20: $100"]:::top
    end
    
    subgraph TIE["🔄 Included via WITH TIES"]
        direction TB
        D["Row 21: $100"]:::tie
        E["Row 22: $100"]:::tie
    end
    
    subgraph EXCLUDED["❌ Excluded"]
        direction TB
        F["Row 23: $90"]:::cut
    end
    
    C -.->|Same Value| D
    D -.->|Same Value| E
    C -->|Cut-off| F
```


***

**Example: TOP WITH TIES and PERCENT**
```sql
-- TOP WITH TIES: Includes products that tie with the 20th row's price
SELECT TOP (20) WITH TIES Name, ListPrice
FROM SalesLT.Product
ORDER BY ListPrice DESC;

-- TOP PERCENT WITH TIES: Returns the top 20% of products by price
SELECT TOP (20) PERCENT WITH TIES Name, ListPrice
FROM SalesLT.Product
ORDER BY ListPrice DESC;
```

***

## 3. Retrieving Pages of Results

`OFFSET-FETCH` is the modern, ANSI-standard way to implement pagination. It allows you to skip a certain number of rows (`OFFSET`) and then return a specific number of rows (`FETCH`).

```mermaid
flowchart LR
    classDef skip fill:#ffcdd2,stroke:#c62828,stroke-width:2px;
    classDef keep fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px;
    classDef ignore fill:#f5f5f5,stroke:#9e9e9e,stroke-width:1px,stroke-dasharray: 5 5;

    subgraph Page 1
        direction LR
        P1_1["Row 1"]:::keep
        P1_2["Row 2"]:::keep
        P1_3["Row 3"]:::keep
    end

    subgraph Page 2
        direction LR
        P2_1["Row 4"]:::skip
        P2_2["Row 5"]:::skip
        P2_3["Row 6"]:::skip
        P2_4["Row 7"]:::keep
        P2_5["Row 8"]:::keep
        P2_6["Row 9"]:::keep
        P2_7["Row 10..."]:::ignore
    end
    
    note1["OFFSET 0, FETCH 3"] -.-> Page 1
    note2["OFFSET 3, FETCH 3"] -.-> Page 2
```


***

**OFFSET-FETCH Examples**
```sql
-- PAGE 1: Start at the beginning, get 10 rows
SELECT Name, ListPrice
FROM SalesLT.Product
ORDER BY Name 
OFFSET 0 ROWS 
FETCH NEXT 10 ROWS ONLY;

-- PAGE 2: Skip the first 10 rows, get the next 10 rows
SELECT Name, ListPrice
FROM SalesLT.Product
ORDER BY Name 
OFFSET 10 ROWS 
FETCH NEXT 10 ROWS ONLY;
```

***

## 4. Removing Duplicates with `DISTINCT`

When selecting a subset of columns, you may end up with duplicate rows. 
- **`ALL`** (Default): Returns every row, including duplicates.
- **`DISTINCT`**: Filters the result set to return only unique combinations of the selected columns.

***

**DISTINCT Examples**
```sql
-- Implicit ALL (Returns duplicates)
SELECT Color
FROM SalesLT.Product;

-- Explicit ALL (Same as above)
SELECT ALL Color
FROM SalesLT.Product;

-- DISTINCT (Returns only unique colors)
SELECT DISTINCT Color
FROM SalesLT.Product;

-- DISTINCT on multiple columns (Returns unique combinations of Color AND Size)
SELECT DISTINCT Color, Size
FROM SalesLT.Product;
```

***

## 5. Filtering Data with the `WHERE` Clause

The `WHERE` clause filters rows based on predicates. Only rows that evaluate to **TRUE** are returned.

### Basic Comparison Operators:
- `=` (Equals)
- `<>` (Not equals)
- `>` (Greater than)

***

**Example: Basic WHERE Clauses**
```sql
-- Equals (=)
SELECT Name, Color, Size
FROM SalesLT.Product
WHERE ProductModelID = 6
ORDER BY Name;

-- Not Equals (<>)
SELECT Name, Color, Size
FROM SalesLT.Product
WHERE ProductModelID <> 6
ORDER BY Name;

-- Greater Than (>)
SELECT Name, ListPrice
FROM SalesLT.Product
WHERE ListPrice > 1000.00
ORDER BY ListPrice;
```

***

### Pattern Matching with `LIKE` and Wildcards

The `LIKE` operator is used for string pattern matching.

| Wildcard | Description | Example |
| :---: | :--- | :--- |
| **`%`** | Any string of zero or more characters | `'HL Road Frame %'` |
| **`_`** | Exactly one single character | `'FR-_[0-9][0-9]_-[0-9][0-9]'` |
| **`[ ]`** | Any single character within the specified range | `'[0-9]'` (Any digit) |
| **`[^ ]`** | Any single character **NOT** within the specified range | `'[^R]'` (Any character except 'R') |

***

**LIKE Operator Examples**
```sql
-- 1. % Wildcard: Starts with 'HL Road Frame ' followed by anything
SELECT Name, ListPrice
FROM SalesLT.Product
WHERE Name LIKE 'HL Road Frame %';

-- 2. Complex Pattern: FR- (any char) (2 digits) - (any char) (2 digits)
-- Example match: FR-X12Y-34
SELECT Name, ListPrice
FROM SalesLT.Product
WHERE ProductNumber LIKE 'FR-_[0-9][0-9]_-[0-9][0-9]';
```

***

### Advanced Filtering Operators

- **`IS NULL` / `IS NOT NULL`**: Checks for missing data. (Never use `= NULL`).
- **`BETWEEN`**: Inclusive range check. Great for dates and numbers.
- **`IN`**: Checks if a value matches any value in a list.
- **`AND` / `OR`**: Combines multiple conditions.

***

**Advanced Filtering Examples**
```sql
-- IS NOT NULL
SELECT Name, ListPrice
FROM SalesLT.Product
WHERE SellEndDate IS NOT NULL;

-- BETWEEN (Inclusive Date Range)
SELECT Name
FROM SalesLT.Product
WHERE SellEndDate BETWEEN '2006/1/1' AND '2006/12/31';

-- IN (List of values)
SELECT ProductCategoryID, Name, ListPrice
FROM SalesLT.Product
WHERE ProductCategoryID IN (5, 6, 7);

-- AND (Both conditions must be true)
SELECT ProductCategoryID, Name, ListPrice, SellEndDate
FROM SalesLT.Product
WHERE ProductCategoryID IN (5, 6, 7) AND SellEndDate IS NULL;

-- OR (At least one condition must be true)
SELECT Name, ProductCategoryID, ProductNumber
FROM SalesLT.Product
WHERE ProductNumber LIKE 'FR%' OR ProductCategoryID IN (5, 6, 7);
```

***

## 🏆 Practice Challenges

Put your knowledge to the test! Try to write the queries for the following scenarios before looking at the solutions in the next cell.

### Challenge 1: Transportation Reports
1. **List of Cities**: Retrieve a list of unique cities and state provinces from the `SalesLT.Address` table, sorted alphabetically by city.
2. **Heaviest Products**: Retrieve the names of the top 10% of products by weight (include ties).

### Challenge 2: Product Data Reports
1. **Model 1 Details**: Get the Name, Color, and Size of products with `ProductModelID = 1`.
2. **Color & Size Filter**: Get the ProductNumber and Name of products that are Black, Red, or White AND are size S or M.
3. **Bike Prefix**: Get the ProductNumber, Name, and ListPrice of products whose ProductNumber starts with `BK-`.
4. **Specific Bike Pattern**: Modify the previous query to find products starting with `BK-`, followed by any character **EXCEPT** 'R', then any characters, ending with a `-` and exactly two numbers.

***

```sql
-- ==========================================
-- CHALLENGE 1 SOLUTIONS
-- ==========================================

-- 1.1 Retrieve a list of unique cities, sorted
SELECT DISTINCT City, StateProvince
FROM SalesLT.Address
ORDER BY City;

-- 1.2 Retrieve the heaviest products (Top 10%)
SELECT TOP (10) PERCENT WITH TIES Name
FROM SalesLT.Product
ORDER BY Weight DESC;


-- ==========================================
-- CHALLENGE 2 SOLUTIONS
-- ==========================================

-- 2.1 Retrieve product details for product model 1
SELECT Name, Color, Size
FROM SalesLT.Product
WHERE ProductModelID = 1;

-- 2.2 Filter products by color and size
SELECT ProductNumber, Name
FROM SalesLT.Product
WHERE Color IN ('Black', 'Red', 'White') 
  AND Size IN ('S', 'M');

-- 2.3 Filter products by product number prefix
SELECT ProductNumber, Name, ListPrice
FROM SalesLT.Product
WHERE ProductNumber LIKE 'BK-%';

-- 2.4 Retrieve specific products by product number
-- Starts with BK-, next char is NOT R, followed by anything, 
-- ends with - and two digits.
SELECT ProductNumber, Name, ListPrice
FROM SalesLT.Product
WHERE ProductNumber LIKE 'BK-[^R]%-[0-9][0-9]';
```

# Module Assessment: Questions, Options, and Solutions

## Question 1
**Question:** You write a Transact-SQL query to list the available sizes for products. Each individual size should be listed only once. Which query should you use?

**Options:**
1. `SELECT Size FROM Production.Product;`
2. `SELECT DISTINCT Size FROM Production.Product;`
3. `SELECT ALL Size FROM Production.Product;`

**✅ Correct Solution:** 
```sql
SELECT DISTINCT Size FROM Production.Product;
```
**Explanation:** The `DISTINCT` keyword removes duplicate rows from the result set, ensuring that each individual size is listed only once. Selecting without `DISTINCT` (or using `ALL`, which is the implicit default) will return duplicate sizes if multiple products share the same size.

---

## Question 2
**Question:** You must return the `InvoiceNo` and `TotalDue` columns from the `Sales.Invoice` table in decreasing order of `TotalDue` value. Which query should you use?

**Options:**
1. `SELECT * FROM Sales.Invoice ORDER BY TotalDue, InvoiceNo;`
2. `SELECT InvoiceNo, TotalDue FROM Sales.Invoice ORDER BY TotalDue DESC;`
3. `SELECT TotalDue AS DESC, InvoiceNo FROM Sales.Invoice;`

**✅ Correct Solution:** 
```sql
SELECT InvoiceNo, TotalDue FROM Sales.Invoice ORDER BY TotalDue DESC;
```
**Explanation:** This query specifically selects the two required columns (`InvoiceNo` and `TotalDue`) and uses `ORDER BY TotalDue DESC` to sort the results in decreasing (descending) order. 
* *Option 1* returns all columns (`*`) and sorts in ascending order by default. 
* *Option 3* incorrectly attempts to use the reserved sorting keyword `DESC` as a column alias.

---

## Question 3
**Question:** Complete this query to return only products that have a Category value of 2 or 4: 
`SELECT Name, Price FROM Production.Product` 
Which clause should you add?

**Options:**
1. `ORDER BY Category;`
2. `WHERE Category BETWEEN 2 AND 4;`
3. `WHERE Category IN (2, 4);`

**✅ Correct Solution:** 
```sql
WHERE Category IN (2, 4);
```
**Explanation:** The `IN` operator is used to filter for specific, discrete values (in this case, exactly 2 or 4). 
* *Option 1* (`ORDER BY`) only sorts the data and does not filter out any rows. 
* *Option 2* (`BETWEEN 2 AND 4`) is inclusive and would incorrectly return products with a Category of 3 as well.